# xfig_06 — Modality Radar Chart

Polar/radar chart where each axis = modality importance for a task.
Importance = |ΔAUROC| when that modality is removed (from ablation table).
Longer spoke = that modality contributes more to this task.

One polygon per task. 5 spokes: No BAS, No RESP, No EKG, Cardio only, BAS only.

Idea #6 from `docs/NEW_PLOT_IDEAS.md`.

**Data**: `results/tables/table6_modality.csv`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
EXPLORE_DIR    = NSRR_TOOLS / "results" / "paper_figures" / "explore"
FINAL_OUT      = EXPLORE_DIR / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)
TABLES_DIR     = NSRR_TOOLS / "results" / "tables"

# Add explore utils to path
_nb_dir = EXPLORE_DIR / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.data_explore import (
    set_root, load_analysis, load_analysis_all_k,
    load_heatmap, load_parquets, load_modality_table,
    subject_predictions, subject_correctness_matrix, CONTEXT_TO_MIN, CTX_ORDER,
)
from utils import panels_explore as xp

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

set_root(WORKSPACE_ROOT)
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 7,
})

# ── Constants ──────────────────────────────────────────────────────────────────
MAIN_TASKS = ["sex_binary", "bmi_binary", "age_class",
              "sleep_efficiency_binary", "apnea_binary"]
TASK_LABEL = xp.TASK_LABEL

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND'}")


In [ ]:
# ── Load modality table ───────────────────────────────────────────────────────
mod_df = load_modality_table(NSRR_TOOLS)
print(mod_df.to_string())
print("\nColumns:", mod_df.columns.tolist())

In [ ]:
# Task names in the table vs internal keys — map them
TASKS = MAIN_TASKS

# Map internal task keys to the 'Task' column values in table6_modality.csv
TASK_TABLE_MAP = {
    "sex_binary":                "Sex",
    "apnea_binary":              "Sleep apnea",
    "sleep_efficiency_binary":   "Sleep efficiency",
    "age_class":                 "Age",
    "bmi_binary":                "BMI",
}

fig = plt.figure(figsize=(5.5, 5.5))
ax = fig.add_subplot(111, projection="polar")

handles, labels = xp.modality_radar_panel(ax, mod_df, tasks=TASKS)
ax.legend(handles, [TASK_LABEL.get(t, t) for t in TASKS],
          loc="upper left", bbox_to_anchor=(1.15, 1.1),
          fontsize=7, frameon=False)
ax.set_title("Modality importance: |ΔAUROC| when modality removed",
             fontsize=8, pad=15)
fig.tight_layout()
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
fig.savefig(str(FINAL_OUT / 'xfig_06_modality_radar.pdf'), bbox_inches='tight')
fig.savefig(str(FINAL_OUT / 'xfig_06_modality_radar.png'), dpi=150, bbox_inches='tight')
print('Saved →', FINAL_OUT / 'xfig_06_modality_radar.pdf')